<h2 align='center'>Codebasics ML Course: ML Flow Tutorial</h2>

In [15]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([9000, 1000]))

In [17]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

### Experiment 1: Train Logistic Regression Classifier

In [18]:
log_reg = LogisticRegression(C=1, solver='liblinear')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_log_reg))

              precision    recall  f1-score   support

           0       0.96      0.98      0.97      2700
           1       0.79      0.58      0.67       300

    accuracy                           0.94      3000
   macro avg       0.87      0.78      0.82      3000
weighted avg       0.94      0.94      0.94      3000



### Experiment 2: Train Random Forest Classifier

In [19]:
rf_clf = RandomForestClassifier(n_estimators=30, max_depth=3)
rf_clf.fit(X_train, y_train)
y_pred_rf = rf_clf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98      2700
           1       0.79      0.75      0.77       300

    accuracy                           0.96      3000
   macro avg       0.88      0.87      0.87      3000
weighted avg       0.95      0.96      0.95      3000



### Experiment 3: Train XGBoost

In [20]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train, y_train)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.97      0.98      0.97      2700
           1       0.76      0.70      0.73       300

    accuracy                           0.95      3000
   macro avg       0.86      0.84      0.85      3000
weighted avg       0.95      0.95      0.95      3000



### Experiment 4: Handle class imbalance using SMOTETomek and then Train XGBoost

In [21]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)

np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([6183, 6183]))

In [22]:
xgb_clf = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_clf.fit(X_train_res, y_train_res)
y_pred_xgb = xgb_clf.predict(X_test)
print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.98      0.95      0.97      2700
           1       0.68      0.85      0.76       300

    accuracy                           0.94      3000
   macro avg       0.83      0.90      0.86      3000
weighted avg       0.95      0.94      0.95      3000



<h2 align="center" style="color:blue">Track Experiments Using MLFlow</h2>

In [23]:
models = [
    (
        "Logistic Regression", 
        LogisticRegression(C=1, solver='liblinear'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        RandomForestClassifier(n_estimators=30, max_depth=3), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [24]:
reports = []

for model_name, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [25]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [ ]:
# Initialize MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Anomaly Detection")

for i, element in enumerate(models):
    model_name = element[0]
    model = element[1]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_param("model", model_name)
        mlflow.log_metric('accuracy', report['accuracy'])
        mlflow.log_metric('recall_class_1', report['1']['recall'])
        mlflow.log_metric('recall_class_0', report['0']['recall'])
        mlflow.log_metric('f1_score_macro', report['macro avg']['f1-score'])        
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2026/04/20 04:11:15 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly Detection2' does not exist. Creating a new experiment.
2026/04/20 04:11:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 04:11:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/20 04:11:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/20 04:11:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deser

🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/3/runs/cebf3ab3099745c699510e498c5f1fdb
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/04/20 04:11:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/3/runs/f3b109fc70554d669d3a2665e264a228
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/04/20 04:11:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/3/runs/2b0a3ad2319a430997bb2f6a47670ace
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/3/runs/9b7b4d0aaeb14aa59e839b97d73a5fac
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
